### Title: RNN Encoder-Decoder Model for Statistical Machine Translation

#### Objectives:
* To build a Sequence-to-Sequence (Seq2Seq) model using RNNs for machine translation.
* To preprocess bilingual text data by tokenizing sentences, creating vocabularies, and converting text into sequences.
* To train and test an encoder-decoder model while understanding embeddings, hidden states, and teacher forcing.


#### Theory:

The Encoder-Decoder or Seq2Seq model is a neural network architecture used to convert one sequence into another, such as translating text from one language to another. It is commonly used in machine translation, text summarization, and speech recognition.

Unlike traditional models, Seq2Seq models can handle input and output sequences of different lengths.

The architecture consists of two main parts:

1. Encoder
The encoder reads the input sentence and converts it into a hidden representation that contains important information about the input sequence.

2. Decoder
The decoder uses the information from the encoder to generate the output sentence word by word. It uses previous outputs and techniques like teacher forcing to improve learning accuracy.


#### Working Principle:
The Seq2Seq model operates through the following stages:

* Tokenize the input sentence into individual words.
* Convert each word into a numerical index and embedding vector.
* The encoder processes the input sequence and generates a context vector.
* The context vector is passed to the decoder.
* The decoder starts with the SOS token and predicts one word at a time.
* Each predicted word is used to generate the next word.
* The process stops when the EOS token is produced, completing the translation.

#### Encoder-Decoder Architecture:

```text
Source Sentence
      │
      ▼
Tokenization
      │
      ▼
Encoder (Embedding + RNN)
      │
      ▼
Context Vector (Hidden State)
      │
      ▼
Decoder (Embedding + RNN)
      │
      ▼
Predicted Sentence
```

## Encoder-Decoder Model Stages

| Stage | Purpose |
|-------|---------|
| Input Sentence | The original sentence given to the model that needs to be translated or processed. |
| Tokenization | Breaks the sentence into smaller units called words or tokens. |
| Word Indices | Converts each token into a numerical value based on the vocabulary. |
| Embedding Layer | Transforms word indices into dense vector representations. |
| Dense Word Vectors | Represents words as continuous vectors containing semantic and language information. |
| Encoder RNN | Processes the word vectors and captures important information from the input sequence. |
| Context Vector | A summarized representation of the input sequence generated by the encoder. |
| Decoder RNN | Uses the context vector to generate the output sequence step by step. |
| Output Sentence | The final translated or predicted sentence generated by the model. |

#### Teacher Forcing:
Teacher Forcing is a training technique used in Seq2Seq models where the decoder receives the correct target word from the training data as the next input instead of its own previous prediction. This helps the model learn faster, improves training accuracy, and reduces error propagation. During testing, teacher forcing is not used because the correct target sequence is unavailable.

Example of Teacher Forcing

Target sentence: SOS → I → am → happy → EOS

With Teacher Forcing: SOS → I → am → happy (Uses the correct previous word as input.) Without Teacher Forcing: SOS → I → am → ... (Uses its own previous prediction as input.)

#### Context Vector:
The Context Vector is the final hidden state produced by the encoder after processing the entire input sequence. It contains the key information and overall meaning of the input sentence, which is then passed to the decoder to generate the output sequence. A more informative context vector generally results in more accurate translations or predictions. Example:

Input: I am happy

Encoder → Context Vector → Decoder

The context vector stores the meaning of "I am happy", enabling the decoder to generate the translated sentence, e.g., "Je suis heureux."

## Word Embeddings

Neural networks cannot understand words directly, so words are first converted into numerical IDs. These IDs are then transformed into dense vector representations using an embedding layer. Word embeddings help the model understand relationships and similarities between words by capturing their semantic meaning.

| Word | One-Hot Representation | Embedding Vector |
|------|------------------------|------------------|
| Cat | [0,1,0] | [0.25, -0.81, 0.63] |
| Dog | [1,0,0] | [0.27, -0.79, 0.60] |

The embedding vectors of Cat and Dog are close to each other because they have similar meanings and are semantically related.

#### Loss Function:
The model evaluates its predictions using Negative Log-Likelihood Loss (NLLLoss) by comparing the predicted output sequence with the target sequence. The loss value indicates the prediction error, where a lower loss signifies better model performance and more accurate translations. Example:

Target: I am happy

Prediction: I am sad

Since the predicted sentence differs from the target sentence, the NLLLoss is high. If the prediction is I am happy, the loss becomes low, indicating better model performance.

#### Applications:
Sequence-to-Sequence models are widely used in:

* Machine Translation
* Chatbots
* Text Summarization
* Speech Recognition
* Question Answering Systems
* Language Generation
* Conversational AI

#### Why Not Feed One-Hot Vectors Directly into an RNN?
A common question in Natural Language Processing (NLP) is why an embedding layer is used before an RNN when words can already be represented as one-hot vectors.

Although an RNN can process one-hot vectors directly, they are inefficient because they are high-dimensional, sparse, and contain no information about the relationships between words. An embedding layer transforms these vectors into compact, dense representations that are easier for the model to learn from.

In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [ ]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [ ]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [ ]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [ ]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis un couard', 'i m a coward']


15

#### Encoder:
The encoder receives the input sentence one token at a time. Each word is first converted into an embedding vector before being processed by the Recurrent Neural Network (RNN). At every time step, the hidden state is updated to retain information from the current and previous words.

After the final input token has been processed, the encoder produces a final hidden state called the context vector, which summarizes the meaning of the entire input sentence.

The hidden state is computed as:

ht = f(xt,ht-1)

where,

xt = input token at time step t ht-1 = previous hidden state ht = updated hidden state f = recurrent function (RNN)

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

#### Decoder:
The decoder generates the output sentence one word at a time.

It begins with a special Start-of-Sequence (SOS) token and uses the encoder's context vector as its initial hidden state. At each step, the decoder predicts the next word in the sequence. The generated word is then fed back into the decoder until the End-of-Sequence (EOS) token is produced or the maximum sequence length is reached.

#### Decoder During Inference:
During inference (testing), the target output is unknown. Hence, the decoder uses the word it predicted in the previous step as the input for the next step. This process continues until the <EOS> (End-of-Sequence) token is generated.

#### Why topk(1)?
The decoder outputs a probability score for every word in the vocabulary.

_, topi = decoder_output.topk(1)
The topk(1) function selects the word with the highest probability, and its index is passed as the next input to the decoder.

#### Why detach()?
decoder_input = topi.squeeze(-1).detach()
The detach() function separates the predicted token from the computation graph, preventing gradient calculations during inference. This reduces memory usage and improves execution speed.

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

        self.softmax = nn.LogSoftmax(dim=-1)

    def forward_step(self, input, hidden):
        output = self.embedding(input)

        output = torch.relu(output)

        output, hidden = self.gru(output, hidden)

        output = self.softmax(self.out(output))

        return output, hidden


    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.empty(
            batch_size, 1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)

        decoder_hidden = encoder_hidden

        decoder_outputs = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden = self.forward_step(
                decoder_input,
                decoder_hidden
            )

            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)

            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()


        decoder_outputs = torch.cat(decoder_outputs, dim=1)

        return decoder_outputs, decoder_hidden, None

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [ ]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [ ]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [ ]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [ ]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)

# Put decoder creation here
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967


In [ ]:
train(
    train_dataloader,
    encoder,
    decoder,
    EPOCHS,
    print_every=5,
    plot_every=5
)

0m 9s (- 6m 1s) (5 2%) 1.9872
0m 17s (- 5m 35s) (10 5%) 1.1960
0m 25s (- 5m 14s) (15 7%) 0.9278
0m 33s (- 5m 5s) (20 10%) 0.7245
0m 41s (- 4m 53s) (25 12%) 0.5606
0m 50s (- 4m 43s) (30 15%) 0.4322
0m 58s (- 4m 34s) (35 17%) 0.3341
1m 7s (- 4m 29s) (40 20%) 0.2586
1m 16s (- 4m 22s) (45 22%) 0.2026
1m 25s (- 4m 16s) (50 25%) 0.1594
1m 34s (- 4m 8s) (55 27%) 0.1305
1m 43s (- 4m 0s) (60 30%) 0.1087
1m 51s (- 3m 51s) (65 32%) 0.0916
1m 59s (- 3m 42s) (70 35%) 0.0804
2m 7s (- 3m 32s) (75 37%) 0.0739
2m 15s (- 3m 23s) (80 40%) 0.0677
2m 24s (- 3m 15s) (85 42%) 0.0636
2m 33s (- 3m 7s) (90 45%) 0.0611
2m 41s (- 2m 58s) (95 47%) 0.0574
2m 49s (- 2m 49s) (100 50%) 0.0567
2m 56s (- 2m 40s) (105 52%) 0.0542
3m 4s (- 2m 31s) (110 55%) 0.0540
3m 12s (- 2m 22s) (115 57%) 0.0490
3m 20s (- 2m 13s) (120 60%) 0.0506
3m 28s (- 2m 5s) (125 62%) 0.0508
3m 37s (- 1m 56s) (130 65%) 0.0492
3m 45s (- 1m 48s) (135 67%) 0.0472
3m 54s (- 1m 40s) (140 70%) 0.0462
4m 3s (- 1m 32s) (145 72%) 0.0475
4m 11s (- 1m 23s) (

In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis encore marie
= i m still married
< i m still married <EOS>

> vous etes tres contraries
= you re very upset
< you re very upset <EOS>

> je me sens confuse
= i m feeling confused
< i m feeling confused <EOS>

> je suis tatillon
= i m fussy
< we re too close <EOS>

> je suis completement epuise
= i m completely exhausted
< i m completely exhausted <EOS>

> nous sommes trop occupes
= we re too busy
< we re too busy <EOS>

> je suis votre chef
= i m your boss
< i m your boss <EOS>

> je me deshabille
= i am undressing
< you re totally ignorant <EOS>

> vous etes insupportable
= you are impossible
< i m gaining weight <EOS>

> elles sont miennes
= they re mine
< they re in prison <EOS>



#### Discussion:

The Seq2Seq RNN Encoder-Decoder model was successfully developed for machine translation. The encoder converted the input sentence into a context vector, and the decoder used this information to generate the translated sentence. The decrease in training loss showed that the model was learning the patterns between the two languages. Teacher forcing helped the model learn faster and make fewer mistakes during training. However, the model worked better for short sentences and had difficulty handling longer sentences because it used a fixed-size context vector.

#### Conclusion:

The RNN Encoder-Decoder model was successfully implemented for sequence translation. The process included data preprocessing, creating a vocabulary, training the model, and generating translations. The model gave good results for simple sentences, but techniques like LSTM, GRU, Attention, and Transformers can improve performance for more complex translations. This implementation helped in understanding the basics of sequence-to-sequence models and machine translation.